In [7]:
from datetime import datetime
from typing import Union
from pathlib import Path
from io import StringIO
import pandas as pd
import requests

def existing_data(file: Union[Path] = "dataset/ohlcv.csv")-> pd.DataFrame():
    if Path(file).exists():
        return pd.read_csv(file)
    return pd.DataFrame()

class collection:
    def __init__(self):
        ...
        
    def collect(url: Union[str] = "https://dse.co.tz/")-> pd.DataFrame:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        tables   = pd.read_html(StringIO(response.text))
        new_data = tables[3].copy()
        today    = datetime.now().strftime("%Y-%m-%d")
        new_data["Date"] = today
        return new_data
        


class cleaning:
    def __init__(self, df: pd.DataFrame):
        ...
    def cleaner(df: pd.DataFrame)-> pd.DataFrame:
        df["Symbol"] = (df["Symbol"].astype(str).str.strip().str.upper())
        df["Date"]   = pd.to_datetime(df["Date"], errors="coerce")
        df["Change"] = (df["Change"]
                        .astype("string")
                        .str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False)
                        .astype(float)
                       )
        df["% Change"] = ((df["Change"]/100) * df['Close'])
        
        df           = df.rename(columns={"Symbol": "Ticker", "Turn over": "Turnover", "MCAP (TZS 'B)": "mktcap"})
        df           = df.rename(columns={"Out Standing Bid": "Bids", "Out Standing Offer": "Offers"})
        df           = df.rename(columns={col: col.lower() for col in df.columns})
        df           = df.drop_duplicates(inplace=False)
        
        for col in df.columns.difference(["date", "ticker", "change"]):
            if col in df.columns:
                df[col] = (
                    df[col]
                    .astype(str)
                    .str.replace(",", "", regex=False)
                    .str.replace("TZS", "", regex=False)
                    .str.strip()
                    .pipe(pd.to_numeric, errors="coerce")
                 )

        columns = ['date', 'ticker','high', 'low', 'open', 
                   'prev close', 'close', 'change', '% change',
                   'volume', 'turnover', 'deals', 'bids', 
                   'offers', 'mktcap'
                    ]
        
        return df[columns].copy()

def main(filepath = "dataset/cleaned.csv"):
    new_data        = collection.collect()
    old_data        = existing_data()
    old_cleaned     = cleaning.cleaner(old_data)
    new_cleaned     = cleaning.cleaner(new_data)
    combined        = pd.concat([old_cleaned, new_cleaned], ignore_index=True)
    combined        = (combined.drop_duplicates(
                            subset=["date", "ticker"],
                            keep="last")
                            .sort_values(["date", "ticker"])
                            .reset_index(drop=True)
                          )
    
    combined.to_csv(filepath, index=False)
    
if __name__ == "__main__":
    main()

In [8]:
new_data        = collection.collect()
old_data        = existing_data()
old_cleaned     = cleaning.cleaner(old_data)
new_cleaned     = cleaning.cleaner(new_data)
combined        = pd.concat([old_cleaned, new_cleaned], ignore_index=True)
combined        = (combined.drop_duplicates(
                        subset=["date", "ticker"],
                        keep="last")
                        .sort_values(["date", "ticker"])
                        .reset_index(drop=True)
                      )


combined.groupby("ticker").tail(1)

,date,ticker,high,low,open,prev close,close,change,% change,volume,turnover,deals,bids,offers,mktcap
300,2026-08-22,AFRIPRISE,625,610,620,620,620,0.00,0.0000,85155,52669260,216,46209,41126,90.5
301,2026-08-22,CRDB,2710,2650,2690,2690,2690,0.00,0.0000,301912,811879890,996,170488,179303,7025.9
302,2026-08-22,DCB,540,495,555,505,505,-9.01,-45.5005,40617,20436890,178,8053,108193,96.8
303,2026-08-22,DSE,6570,6340,6340,6480,6480,2.21,143.2080,1220,7946810,29,2589,9184,154.4
304,2026-08-22,EABL,0,0,5710,0,5710,0.00,0.0000,0,0,0,0,0,4515.3
305,2026-08-22,IEACLC-ETF,1380,1360,1360,1370,1370,0.74,10.1380,39790,54644290,63,182914,70797,193.3
306,2026-08-22,JATU,280,280,270,270,270,0.00,0.0000,40,11200,3,160,0,5.4
307,2026-08-22,JHL,0,0,8650,0,8650,0.00,0.0000,0,0,0,454,0,626.9
308,2026-08-22,KA,0,0,115,0,115,0.00,0.0000,0,0,0,13,0,653.4
309,2026-08-22,KCB,1980,1960,1970,1970,1970,0.00,0.0000,17060,33626880,42,37668,9282,5851.6
